# **NLP701@MBZUAI Fall 2025 - Lab 14**

## **Learning Outcomes**
- You will be able to describe the core concept of BERT, one of the most influential models in NLP.
- You will be able to fine-tune BERT for NER and evaluate it.
- You will be able to fine-tune a varient of BERT and compare the results.

## **Learning Activities**
- Fine-tune BERT for NER and evaluate.
- Pick a variant of BERT and fine-tune it.
- Compare the results (in addition to comparing it with the results in the paper).

If you're opening this Notebook on colab, you will probably need to install 🤗 Transformers and 🤗 Datasets. Uncomment the following cell and run it.

In [1]:
!pip install datasets transformers[torch] seqeval accelerate evaluate

# **Fine-tuning a BERT model on a token classification task (NER)**

<div>
<img src="https://drive.google.com/uc?export=view&id=1RxpphUbBQON11opxggrzcKOXxyCFlsB1" width="400"/>
</div>

In this notebook, we will see how to fine-tune one of the [🤗 Transformers](https://github.com/huggingface/transformers) model to a token classification task, which is the task of predicting a label for each token.
We will see how to easily load a dataset for these kinds of tasks and use the `Trainer` API to fine-tune a model on it.

This notebook is built to run on any token classification task, with any model checkpoint from the [Model Hub](https://huggingface.co/models) as long as that model has a version with a token classification head and a fast tokenizer (check on [this table](https://huggingface.co/transformers/index.html#bigtable) if this is the case). It might just need some small adjustments if you decide to use a different dataset than the one used here. Depending on you model and the GPU you are using, you might need to adjust the batch size to avoid out-of-memory errors. Set those three parameters, then the rest of the notebook should run smoothly:

In [2]:
task = "ner"
model_checkpoint = "bert-base-cased"
batch_size = 32

## Loading the dataset

We will use the [🤗 Datasets](https://github.com/huggingface/datasets) library to download the data and get the metric we need to use for evaluation (to compare our model to the benchmark). This can be easily done with the functions `load_dataset` and `load_metric`.  

In [3]:
from datasets import load_dataset

/home/khang.nhat/anaconda3/envs/nlplab/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


For our example here, we'll use the [CoNLL 2003 dataset](https://www.aclweb.org/anthology/W03-0419.pdf). The notebook should work with any token classification dataset provided by the 🤗 Datasets library. If you're using your own dataset defined from a JSON or csv file (see the [Datasets documentation](https://huggingface.co/docs/datasets/loading_datasets.html#from-local-files) on how to load them), it might need some adjustments in the names of the columns used.

In [4]:
datasets = load_dataset("lhoestq/conll2003")

The `datasets` object itself is [`DatasetDict`](https://huggingface.co/docs/datasets/package_reference/main_classes.html#datasetdict), which contains one key for the training, validation and test set.

In [5]:
datasets

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

We can see the training, validation and test sets all have a column for the tokens (the input texts split into words) and one column of labels for each kind of task we introduced before.

To access an actual element, you need to select a split first, then give an index:

In [6]:
datasets["train"][0]

{'id': '0',
 'tokens': ['EU',
  'rejects',
  'German',
  'call',
  'to',
  'boycott',
  'British',
  'lamb',
  '.'],
 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7],
 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0],
 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}

The labels are already coded as integer ids to be easily usable by our model, but the correspondence with the actual categories is stored in the `features` of the dataset:

In [7]:
datasets["train"].features[f"ner_tags"]

List(Value('int64'))

So for the NER tags, 0 corresponds to 'O', 1 to 'B-PER' etc... On top of the 'O' (which means no special entity), there are four labels for NER here, each prefixed with 'B-' (for beginning) or 'I-' (for intermediate), that indicate if the token is the first one for the current group with the label or not:
- 'PER' for person
- 'ORG' for organization
- 'LOC' for location
- 'MISC' for miscellaneous

Since the labels are lists of `ClassLabel`, the actual names of the labels are nested in the `feature` attribute of the object above:

In [8]:
label_list = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']
label_list

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

To get a sense of what the data looks like, the following function will show some examples picked randomly in the dataset (automatically decoding the labels in passing).

In [9]:
from datasets import ClassLabel, Sequence
import random
import pandas as pd
from IPython.display import display, HTML

def show_random_elements(dataset, num_examples=10):
    assert num_examples <= len(dataset), "Can't pick more elements than there are in the dataset."
    picks = []
    for _ in range(num_examples):
        pick = random.randint(0, len(dataset)-1)
        while pick in picks:
            pick = random.randint(0, len(dataset)-1)
        picks.append(pick)

    df = pd.DataFrame(dataset[picks])
    for column, typ in dataset.features.items():
        if isinstance(typ, ClassLabel):
            df[column] = df[column].transform(lambda i: typ.names[i])
        elif isinstance(typ, Sequence) and isinstance(typ.feature, ClassLabel):
            df[column] = df[column].transform(lambda x: [typ.feature.names[i] for i in x])
    display(HTML(df.to_html()))

In [10]:
show_random_elements(datasets["train"])

,id,tokens,pos_tags,chunk_tags,ner_tags
0,3978,"[Lille, 3, 2, 0, 1, 4, 3, 6]","[22, 11, 11, 11, 11, 11, 11, 11]","[11, 12, 12, 12, 12, 12, 12, 12]","[3, 0, 0, 0, 0, 0, 0, 0]"
1,2336,"[1., Allen, Johnson, (, U.S., ), 12.92, seconds]","[11, 22, 22, 4, 22, 5, 11, 24]","[11, 12, 12, 0, 11, 0, 11, 12]","[0, 1, 2, 0, 5, 0, 0, 0]"
2,5214,"[Uralmash, Yekaterinburg, 24, 3, 8, 13, 24, 43, 16]","[22, 22, 11, 11, 11, 11, 11, 11, 11]","[11, 12, 12, 12, 12, 12, 12, 12, 12]","[3, 4, 0, 0, 0, 0, 0, 0, 0]"
3,2708,"[But, we, know, that, these, are, only, words, ,, nobody, has, died, from, hearing, them, and, it, only, makes, us, support, our, team, more, vehemently, ., ""]","[10, 28, 41, 15, 12, 41, 30, 24, 6, 21, 42, 40, 15, 39, 28, 10, 28, 30, 42, 28, 41, 29, 21, 31, 30, 7, 0]","[0, 11, 21, 17, 11, 21, 11, 12, 0, 11, 21, 22, 13, 21, 11, 0, 11, 3, 21, 11, 21, 11, 12, 1, 21, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
4,4866,"[10., Neil, Hodgson, (, Britain, ), Ducati, 82]","[11, 22, 22, 4, 22, 5, 22, 11]","[11, 12, 12, 0, 11, 0, 11, 12]","[0, 1, 2, 0, 5, 0, 3, 0]"
5,5879,"[Fitzroy, 21, 1, 0, 20, 1381, 2778, 49.7, 4]","[22, 11, 11, 11, 11, 11, 11, 11, 11]","[11, 12, 12, 12, 12, 12, 12, 12, 12]","[3, 0, 0, 0, 0, 0, 0, 0, 0]"
6,13419,"[Fowler, 's, predecessor, Raymond, Mabus, returned, to, the, United, States, in, May, .]","[22, 27, 21, 22, 22, 38, 35, 12, 22, 23, 15, 22, 7]","[11, 11, 12, 12, 12, 21, 13, 11, 12, 12, 13, 11, 0]","[1, 0, 0, 1, 2, 0, 0, 0, 5, 6, 0, 0, 0]"
7,4811,"[4., Troy, Corser, (, Australia, ), Ducati, 38:34.436]","[11, 22, 22, 4, 22, 5, 22, 11]","[11, 12, 12, 0, 11, 0, 11, 12]","[0, 1, 2, 0, 5, 0, 3, 0]"
8,1736,"[Ka, Wah, Bank, sets, HK$, 43, mln, FRCD, .]","[22, 22, 22, 42, 3, 11, 21, 22, 7]","[11, 12, 12, 21, 11, 12, 12, 12, 0]","[3, 4, 4, 0, 7, 0, 0, 0, 0]"
9,12999,"[The, tabloid, Star, magazine, reported, the, married, Morris, had, a, lengthy, affair, with, a, $, 200-an-hour, prostitute, who, he, allowed, to, eavesdrop, on, telephone, conversations, with, Clinton, ,, and, with, whom, he, shared, White, House, speeches, before, they, were, made, .]","[12, 16, 22, 21, 38, 12, 40, 22, 38, 12, 16, 21, 15, 12, 3, 16, 21, 44, 28, 38, 35, 37, 15, 21, 24, 15, 22, 6, 10, 15, 44, 28, 38, 22, 22, 24, 15, 28, 38, 40, 7]","[11, 12, 12, 12, 21, 11, 12, 12, 21, 11, 12, 12, 13, 11, 12, 12, 12, 11, 11, 21, 22, 22, 13, 11, 12, 13, 11, 0, 0, 13, 11, 11, 21, 11, 12, 12, 17, 11, 21, 22, 0]","[0, 0, 3, 4, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 5, 6, 0, 0, 0, 0, 0, 0]"


## Preprocessing the data

Before we can feed those texts to our model, we need to preprocess them. This is done by a 🤗 Transformers `Tokenizer` which will (as the name indicates) tokenize the inputs (including converting the tokens to their corresponding IDs in the pretrained vocabulary) and put it in a format the model expects, as well as generate the other inputs that model requires.

To do all of this, we instantiate our tokenizer with the `AutoTokenizer.from_pretrained` method, which will ensure:

- we get a tokenizer that corresponds to the model architecture we want to use,
- we download the vocabulary used when pretraining this specific checkpoint.

That vocabulary will be cached, so it's not downloaded again the next time we run the cell.

In [11]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

The following assertion ensures that our tokenizer is a fast tokenizers (backed by Rust) from the 🤗 Tokenizers library. Those fast tokenizers are available for almost all models, and we will need some of the special features they have for our preprocessing.

In [12]:
import transformers
assert isinstance(tokenizer, transformers.PreTrainedTokenizerFast)

You can check which type of models have a fast tokenizer available and which don't on the [big table of models](https://huggingface.co/transformers/index.html#bigtable).

You can directly call this tokenizer on one sentence:

In [13]:
tokenizer("Hello, this is one sentence!")

{'input_ids': [101, 8667, 117, 1142, 1110, 1141, 5650, 106, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}

Depending on the model you selected, you will see different keys in the dictionary returned by the cell above. They don't matter much for what we're doing here (just know they are required by the model we will instantiate later), you can learn more about them in [this tutorial](https://huggingface.co/transformers/preprocessing.html) if you're interested.

If, as is the case here (in CoNLL format where we already have word boundaries), your inputs have already been split into words, you should pass the list of words to your tokenzier with the argument `is_split_into_words=True`:

In [14]:
tokenizer(["Hello", ",", "this", "is", "one", "sentence", "split", "into", "words", "."], is_split_into_words=True)

{'input_ids': [101, 8667, 117, 1142, 1110, 1141, 5650, 3325, 1154, 1734, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

Note that transformers are often pretrained with subword tokenizers, meaning that even if your inputs have been split into words already, each of those words could be split again by the tokenizer. Let's look at an example of that:

In [15]:
example = datasets["train"][4]
print(example["tokens"])

['Germany', "'s", 'representative', 'to', 'the', 'European', 'Union', "'s", 'veterinary', 'committee', 'Werner', 'Zwingmann', 'said', 'on', 'Wednesday', 'consumers', 'should', 'buy', 'sheepmeat', 'from', 'countries', 'other', 'than', 'Britain', 'until', 'the', 'scientific', 'advice', 'was', 'clearer', '.']


In [16]:
tokenized_input = tokenizer(example["tokens"], is_split_into_words=True)
tokens = tokenizer.convert_ids_to_tokens(tokenized_input["input_ids"])
print(tokens)

['[CLS]', 'Germany', "'", 's', 'representative', 'to', 'the', 'European', 'Union', "'", 's', 'veterinary', 'committee', 'Werner', 'Z', '##wing', '##mann', 'said', 'on', 'Wednesday', 'consumers', 'should', 'buy', 'sheep', '##me', '##at', 'from', 'countries', 'other', 'than', 'Britain', 'until', 'the', 'scientific', 'advice', 'was', 'clearer', '.', '[SEP]']


Here the words "Zwingmann" and "sheepmeat" have been split in three subtokens.

This means that we need to do some processing on our labels as the input ids returned by the tokenizer are longer than the lists of labels our dataset contain, first because some special tokens might be added (we can a `[CLS]` and a `[SEP]` above) and then because of those possible splits of words in multiple tokens:

In [17]:
len(example[f"{task}_tags"]), len(tokenized_input["input_ids"])

(31, 39)

Thankfully, the tokenizer returns outputs that have a `word_ids` method which can help us.

In [18]:
print(tokenized_input.word_ids())

[None, 0, 1, 1, 2, 3, 4, 5, 6, 7, 7, 8, 9, 10, 11, 11, 11, 12, 13, 14, 15, 16, 17, 18, 18, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, None]


As we can see, it returns a list with the same number of elements as our processed input ids, mapping special tokens to `None` and all other tokens to their respective word. This way, we can align the labels with the processed input ids.

In [19]:
word_ids = tokenized_input.word_ids()
aligned_labels = [-100 if i is None else example[f"{task}_tags"][i] for i in word_ids]
print(len(aligned_labels), len(tokenized_input["input_ids"]))

39 39


We can confirm how labels are mapped to each subwords. For example, "Zwingman" is tokenized as three subwords, "Z", "##wing", "##mann" and all the subwords are assigned the same label ("I-PER").

In [20]:
df = pd.DataFrame({
     "inputs": tokenizer.convert_ids_to_tokens(tokenized_input["input_ids"]),
     "labels": [None if i==-100 else label_list[i] for i in aligned_labels]
     })
df

,inputs,labels
0,[CLS],NaN
1,Germany,B-LOC
2,',O
3,s,O
4,representative,O
5,to,O
6,the,O
7,European,B-ORG
8,Union,I-ORG
9,',O


Here we set the labels of all special tokens to -100 (the index that is ignored by PyTorch) and the labels of all other tokens to the label of the word they come from. Another strategy is to set the label only on the first token obtained from a given word, and give a label of -100 to the other subtokens from the same word. We propose the two strategies here, just change the value of the following flag:

In [21]:
label_all_tokens = True

We're now ready to write the function that will preprocess our samples. We feed them to the `tokenizer` with the argument `truncation=True` (to truncate texts that are bigger than the maximum size allowed by the model) and `is_split_into_words=True` (as seen above). Then we align the labels with the token ids using the strategy we picked:

In [22]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(examples[f"{task}_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            # Special tokens have a word id that is None. We set the label to -100 so they are automatically
            # ignored in the loss function.
            if word_idx is None:
                label_ids.append(-100)
            # We set the label for the first token of each word.
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            # For the other tokens in a word, we set the label to either the current label or -100, depending on
            # the label_all_tokens flag.
            else:
                label_ids.append(label[word_idx] if label_all_tokens else -100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

This function works with one or several examples. In the case of several examples, the tokenizer will return a list of lists for each key:

In [23]:
tokenize_and_align_labels(datasets['train'][:5])

{'input_ids': [[101, 7270, 22961, 1528, 1840, 1106, 21423, 1418, 2495, 12913, 119, 102], [101, 1943, 14428, 102], [101, 26660, 13329, 12649, 15928, 1820, 118, 4775, 118, 1659, 102], [101, 1109, 1735, 2827, 1163, 1113, 9170, 1122, 19786, 1114, 1528, 5566, 1106, 11060, 1106, 188, 17315, 1418, 2495, 12913, 1235, 6479, 4959, 2480, 6340, 13991, 3653, 1169, 1129, 12086, 1106, 8892, 119, 102], [101, 1860, 112, 188, 4702, 1106, 1103, 1735, 1913, 112, 188, 27431, 3914, 14651, 163, 7635, 4119, 1163, 1113, 9031, 11060, 1431, 4417, 8892, 3263, 2980, 1121, 2182, 1168, 1190, 2855, 1235, 1103, 3812, 5566, 1108, 27830, 119, 102]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 

To apply this function on all the sentences (or pairs of sentences) in our dataset, we just use the `map` method of our `dataset` object we created earlier. This will apply the function on all the elements of all the splits in `dataset`, so our training, validation and testing data will be preprocessed in one single command.

In [24]:
tokenized_datasets = datasets.map(tokenize_and_align_labels, batched=True)

Even better, the results are automatically cached by the 🤗 Datasets library to avoid spending time on this step the next time you run your notebook. The 🤗 Datasets library is normally smart enough to detect when the function you pass to map has changed (and thus requires to not use the cache data). For instance, it will properly detect if you change the task in the first cell and rerun the notebook. 🤗 Datasets warns you when it uses cached files, you can pass `load_from_cache_file=False` in the call to `map` to not use the cached files and force the preprocessing to be applied again.

Note that we passed `batched=True` to encode the texts by batches together. This is to leverage the full benefit of the fast tokenizer we loaded earlier, which will use multi-threading to treat the texts in a batch concurrently.

## Fine-tuning the model

Now that our data is ready, we can download the pretrained model and fine-tune it. Since all our tasks are about token classification, we use the `AutoModelForTokenClassification` class. Like with the tokenizer, the `from_pretrained` method will download and cache the model for us. The only thing we have to specify is the number of labels for our problem (which we can get from the features, as seen before):

In [25]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer

model = AutoModelForTokenClassification.from_pretrained(model_checkpoint, num_labels=len(label_list))

A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Pl

In case there is a warning, this warning is telling us we are throwing away some weights (the `vocab_transform` and `vocab_layer_norm` layers) and randomly initializing some other (the `pre_classifier` and `classifier` layers). This is absolutely normal in this case, because we are removing the head used to pretrain the model on a masked language modeling objective and replacing it with a new head for which we don't have pretrained weights, so the library warns us we should fine-tune this model before using it for inference, which is exactly what we are going to do.

To instantiate a `Trainer`, we will need to define three more things. The most important is the [`TrainingArguments`](https://huggingface.co/transformers/main_classes/trainer.html#transformers.TrainingArguments), which is a class that contains all the attributes to customize the training. It requires one folder name, which will be used to save the checkpoints of the model, and all other arguments are optional:

In [26]:
random_seed = 12345
model_name = model_checkpoint.split("/")[-1]
args = TrainingArguments(
    f"{model_name}-finetuned-{task}-seed-{str(random_seed)}",
    eval_strategy = "epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=3,
    weight_decay=0.01,
    seed=random_seed
)

Here we set the evaluation to be done at the end of each epoch, tweak the learning rate, use the `batch_size` defined at the top of the notebook and customize the number of epochs for training, as well as the weight decay.
Then we will need a data collator that will batch our processed examples together while applying padding to make them all the same size (each pad will be padded to the length of its longest example). There is a data collator for this task in the Transformers library, that not only pads the inputs, but also the labels:

In [27]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

The last thing to define for our `Trainer` is how to compute the metrics from the predictions. Here we will load the [`seqeval`](https://github.com/chakki-works/seqeval) metric (which is commonly used to evaluate results on the CoNLL dataset) via the Datasets library.

In [28]:
import evaluate
metric = evaluate.load("seqeval")

This metric takes list of labels for the predictions and references:

In [29]:
labels = [label_list[i] for i in example[f"{task}_tags"]]
metric.compute(predictions=[labels], references=[labels])

{'LOC': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 2},
 'ORG': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 1},
 'PER': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 1},
 'overall_precision': 1.0,
 'overall_recall': 1.0,
 'overall_f1': 1.0,
 'overall_accuracy': 1.0}

So we will need to do a bit of post-processing on our predictions:
- select the predicted index (with the maximum logit) for each token
- convert it to its string label
- ignore everywhere we set a label of -100

The following function does all this post-processing on the result of `Trainer.evaluate` (which is a namedtuple containing predictions and labels) before applying the metric:

In [30]:
import numpy as np

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (special tokens)
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

Note that we drop the precision/recall/f1 computed for each category and only focus on the overall precision/recall/f1/accuracy.

Then we just need to pass all of this along with our datasets to the `Trainer`:

In [31]:
trainer = Trainer(
    model,
    args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

We can now finetune our model by just calling the `train` method.
While training, have a look at the BERT paper (by the way, the paper won the Best Paper Award at NAACL2019). Were you able to reproduce their result in Table 7?
- https://aclanthology.org/N19-1423.pdf

In [32]:
trainer.train()

/home/khang.nhat/anaconda3/envs/nlplab/lib/python3.11/site-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.070029,0.911686,0.919702,0.915677,0.979396
2,No log,0.057307,0.933232,0.936749,0.934987,0.984164
3,0.112000,0.055030,0.936733,0.944464,0.940583,0.985253


/home/khang.nhat/anaconda3/envs/nlplab/lib/python3.11/site-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/home/khang.nhat/anaconda3/envs/nlplab/lib/python3.11/site-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


TrainOutput(global_step=660, training_loss=0.09098203182220459, metrics={'train_runtime': 152.4686, 'train_samples_per_second': 276.273, 'train_steps_per_second': 4.329, 'total_flos': 1279585869025536.0, 'train_loss': 0.09098203182220459, 'epoch': 3.0})

The `evaluate` method allows you to evaluate again on the evaluation dataset or on another dataset:

In [33]:
trainer.evaluate()

{'eval_loss': 0.055030275136232376,
 'eval_precision': 0.9367325146823278,
 'eval_recall': 0.9444643818410192,
 'eval_f1': 0.9405825589706933,
 'eval_accuracy': 0.9852534290928358,
 'eval_runtime': 5.0643,
 'eval_samples_per_second': 641.751,
 'eval_steps_per_second': 10.071,
 'epoch': 3.0}

To get the precision/recall/f1 computed for each category now that we have finished training, we can apply the same function as before on the result of the `predict` method:

In [34]:
predictions, labels, _ = trainer.predict(tokenized_datasets["validation"])
predictions = np.argmax(predictions, axis=2)

# Remove ignored index (special tokens)
true_predictions = [
    [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]
true_labels = [
    [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]

results = metric.compute(predictions=true_predictions, references=true_labels)
results

{'LOC': {'precision': 0.9661768782922096,
  'recall': 0.9587345254470426,
  'f1': 0.9624413145539906,
  'number': 3635},
 'MISC': {'precision': 0.8499673842139596,
  'recall': 0.8804054054054054,
  'f1': 0.8649186856953203,
  'number': 1480},
 'ORG': {'precision': 0.9191770756796473,
  'recall': 0.9259807549962991,
  'f1': 0.9225663716814159,
  'number': 2702},
 'PER': {'precision': 0.9588270142180095,
  'recall': 0.9723640732952838,
  'f1': 0.9655480984340045,
  'number': 3329},
 'overall_precision': 0.9367325146823278,
 'overall_recall': 0.9444643818410192,
 'overall_f1': 0.9405825589706933,
 'overall_accuracy': 0.9852534290928358}

# **Exercise: Pick a variant of BERT, fine-tune it, evaluate, and compare with BERT fine-tuned model.**

- In the class, we have covered variants of BERT models, such as RoBERTa, SpanBERT, ALBERT, etc.
- Go to the [Model Hub](https://huggingface.co/models) and pick one of the BERT variants, and fine-tune it following the same experimental setup.
- Evaluate the model and describe the experimental results, comparing with the BERT fine-tuned model.

### **Fine-tuning RoBERTa for NER**

In [35]:
# Define model checkpoint for RoBERTa
roberta_model_checkpoint = "roberta-base"

# Initialize RoBERTa Tokenizer
roberta_tokenizer = AutoTokenizer.from_pretrained(roberta_model_checkpoint, add_prefix_space=True)

# Verify fast tokenizer
assert isinstance(roberta_tokenizer, transformers.PreTrainedTokenizerFast)

# Define a new preprocessing function for RoBERTa
def tokenize_and_align_labels_roberta(examples):
    tokenized_inputs = roberta_tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(examples[f"{task}_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(label[word_idx] if label_all_tokens else -100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Tokenize and align labels for RoBERTa
tokenized_datasets_roberta = datasets.map(tokenize_and_align_labels_roberta, batched=True)

Map: 100%|██████████| 3453/3453 [00:00<00:00, 15198.35 examples/s]


In [36]:
# Initialize RoBERTa model for Token Classification
roberta_model = AutoModelForTokenClassification.from_pretrained(roberta_model_checkpoint, num_labels=len(label_list))

# Define TrainingArguments for RoBERTa
roberta_args = TrainingArguments(
    f"{roberta_model_checkpoint.split('/')[-1]}-finetuned-{task}-seed-{str(random_seed)}",
    eval_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=3,
    weight_decay=0.01,
    seed=random_seed,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

# Data collator for RoBERTa
roberta_data_collator = DataCollatorForTokenClassification(roberta_tokenizer)

# Initialize Trainer for RoBERTa
roberta_trainer = Trainer(
    roberta_model,
    roberta_args,
    train_dataset=tokenized_datasets_roberta["train"],
    eval_dataset=tokenized_datasets_roberta["validation"],
    data_collator=roberta_data_collator,
    tokenizer=roberta_tokenizer,
    compute_metrics=compute_metrics
)

# Train RoBERTa model
roberta_trainer.train()

Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/khang.nhat/anaconda3/envs/nlplab/lib/python3.11/site-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.063262,0.922655,0.936249,0.929402,0.982280
2,No log,0.048897,0.950898,0.954760,0.952825,0.988080
3,0.107300,0.047938,0.952734,0.959482,0.956096,0.988801


/home/khang.nhat/anaconda3/envs/nlplab/lib/python3.11/site-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/home/khang.nhat/anaconda3/envs/nlplab/lib/python3.11/site-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/home/khang.nhat/anaconda3/envs/nlplab/lib/python3.11/site-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


TrainOutput(global_step=660, training_loss=0.08794648575060296, metrics={'train_runtime': 173.5655, 'train_samples_per_second': 242.692, 'train_steps_per_second': 3.803, 'total_flos': 1248097617181440.0, 'train_loss': 0.08794648575060296, 'epoch': 3.0})

In [37]:
# Evaluate RoBERTa model
roberta_eval_results = roberta_trainer.evaluate()
print("RoBERTa Evaluation Results:", roberta_eval_results)

# Predict and compute detailed metrics for RoBERTa
roberta_predictions, roberta_labels, _ = roberta_trainer.predict(tokenized_datasets_roberta["validation"])
roberta_predictions = np.argmax(roberta_predictions, axis=2)

roberta_true_predictions = [
    [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(roberta_predictions, roberta_labels)
]
roberta_true_labels = [
    [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(roberta_predictions, roberta_labels)
]

roberta_results = metric.compute(predictions=roberta_true_predictions, references=roberta_true_labels)
print("RoBERTa Detailed Results:", roberta_results)

/home/khang.nhat/anaconda3/envs/nlplab/lib/python3.11/site-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


RoBERTa Evaluation Results: {'eval_loss': 0.04793832451105118, 'eval_precision': 0.9527337522273281, 'eval_recall': 0.9594824329429543, 'eval_f1': 0.9560961837090018, 'eval_accuracy': 0.9888005891196956, 'eval_runtime': 5.6261, 'eval_samples_per_second': 577.66, 'eval_steps_per_second': 9.065, 'epoch': 3.0}
RoBERTa Detailed Results: {'LOC': {'precision': 0.9755065013607499, 'recall': 0.970809509479386, 'f1': 0.9731523378582202, 'number': 3323}, 'MISC': {'precision': 0.8857350800582242, 'recall': 0.8981549815498155, 'f1': 0.891901795529498, 'number': 1355}, 'ORG': {'precision': 0.9399555226093402, 'recall': 0.9491017964071856, 'f1': 0.9445065176908752, 'number': 2672}, 'PER': {'precision': 0.9683313032886723, 'recall': 0.9820877084620135, 'f1': 0.9751609935602575, 'number': 3238}, 'overall_precision': 0.9527337522273281, 'overall_recall': 0.9594824329429543, 'overall_f1': 0.9560961837090018, 'overall_accuracy': 0.9888005891196956}
